In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

In [ ]:
df = pd.read_csv("data_split_normalized_wp.csv")


In [ ]:
AA_LIST = "ACDEFGHIKLMNPQRSTVWY"
AA_DICT = {aa: i for i, aa in enumerate(AA_LIST)}
MAX_LEN = df['mutated Sequence'].apply(len).max()
input_size = MAX_LEN * len(AA_LIST)

def one_hot_encode(seq):
    encoding = torch.zeros(MAX_LEN, len(AA_LIST), dtype=torch.float32)
    for i, aa in enumerate(seq):
        if i >= MAX_LEN:
            break
        if aa in AA_DICT:
            encoding[i, AA_DICT[aa]] = 1.0
    return encoding.flatten()

In [ ]:
class AASeqDataset(Dataset):
    def __init__(self, df):
        self.sequences = df['mutated Sequence'].values
        self.scores = df['total_score_z'].values.astype(float)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        x = one_hot_encode(self.sequences[idx])
        y = torch.tensor(self.scores[idx], dtype=torch.float32)
        return x, y

In [ ]:
train_dataset = AASeqDataset(df[df['split']=='train'])
val_dataset = AASeqDataset(df[df['split']=='val'])
test_dataset = AASeqDataset(df[df['split']=='test'])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:
model = nn.Linear(input_size, 1)


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
x_sample, y_sample = train_dataset[0]
print(x_sample.shape, input_size)  # should print torch.Size([5320]) 5320

print(model.weight.shape)

In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pearson, _ = pearsonr(y_true, y_pred)
    spearman, _ = spearmanr(y_true, y_pred)

    return {
        "MSE": mse, "MAE": mae, "RMSE": rmse,
        "R2": r2, "Pearson": pearson, "Spearman": spearman
    }

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    total_loss = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            y_pred = model(x_batch).squeeze(-1) # Squeeze to match [batch_size]
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()

            all_preds.extend(y_pred.tolist())
            all_targets.extend(y_batch.tolist())

    metrics = calculate_metrics(all_targets, all_preds)
    metrics['Loss'] = total_loss / len(loader)
    return metrics

In [ ]:
print(f"Starting training. Input size: {input_size}")

for epoch in range(1):
    model.train()
    train_loss = 0
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(x_batch).squeeze(-1)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    val_results = evaluate(model, val_loader)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_results['Loss']:.4f}")
    print(f"   Val Spearman: {val_results['Spearman']:.4f} | Val R2: {val_results['R2']:.4f}")

In [ ]:
test_results = evaluate(model, test_loader)
print("\n" + "="*30)
print("FINAL TEST SET METRICS")
print("="*30)
for key, value in test_results.items():
    print(f"{key:10}: {value:.4f}")